Fine-tune the summarizer

Trains a LoRA adapter for the model `finetune.teacher` names in
`config/idhazh.json`, on the corpus this repository already commits. Runs top
to bottom on a free Colab T4. Nothing here needs a paid tier and nothing is
hardcoded to a model.

**Open in Colab, set the runtime to a GPU, then Runtime -> Run all.**

What it does, in order:

1. Checks the GPU and installs what it needs.
2. Clones this repository and reads `config/idhazh.json`.
3. Puts every knob in one cell, so nothing is buried further down.
4. Resolves the teacher's base weights and stops if they are not the ones
   production serves.
5. Loads the corpus and the hand-written reference set, and refuses every row
   it is not allowed to train on.
6. Builds the batch and asserts the loss mask before a single step runs.
7. Trains, checkpointing to Drive so a dropped session costs minutes.
8. Shows you what the tuned model writes, next to what the base wrote.
9. Saves the adapter, with its SHA-256 and byte count.
10. Downloads it, or pushes it to Hugging Face if you give it a token.

**The merge and the quantise happen on your machine, not here.** The last cell
says how. Merging in Colab loads the whole model at 16-bit and wants about
16 GB of ordinary RAM against the free tier's 12, so training would succeed and
the save would die.

The weights are not committed to this repository. This notebook is.

In [ ]:
# ---------------------------------------------------------------------------
# 1. Bootstrap. What machine is this, and does it have what we need?
# ---------------------------------------------------------------------------
# The free T4 is a Turing card: fp16 only, no bf16. A TPU has no QLoRA path at
# all, because 4-bit quantisation is CUDA-only. Both are stops, not warnings.

import json
import os
import platform
import subprocess
import sys
import time

RUN_STARTED = time.time()

IN_COLAB = "google.colab" in sys.modules

print("python  ", platform.python_version())
print("platform", platform.platform())
print("colab   ", IN_COLAB)

# Pinned low, not exact. Colab moves its base image, and an exact pin here turns
# a working notebook into a resolver conflict a month from now. The versions
# that actually ran are printed below and belong in the measurements record.
DEPENDENCIES = [
    "torch>=2.4",
    "transformers>=4.45",
    "peft>=0.13",
    "trl>=0.11",
    "bitsandbytes>=0.44",
    "accelerate>=1.0",
    "datasets>=3.0",
    "huggingface_hub>=0.25",
    "sentencepiece>=0.2",
]

print("\ninstalling, this takes a couple of minutes on a cold runtime ...")
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-U", *DEPENDENCIES],
    check=True,
)

import torch  # noqa: E402 - pip ran above

if not torch.cuda.is_available():
    raise SystemExit(
        "No CUDA device. Runtime -> Change runtime type -> T4 GPU, then run this "
        "cell again. A TPU will not work: 4-bit quantisation is CUDA-only, so "
        "there is no QLoRA path on one."
    )

GPU_NAME = torch.cuda.get_device_name(0)
GPU_VRAM_GB = torch.cuda.get_device_properties(0).total_memory / 1024**3
GPU_CAPABILITY = torch.cuda.get_device_capability(0)
SUPPORTS_BF16 = torch.cuda.is_bf16_supported()

print(f"\ngpu       {GPU_NAME}")
print(f"vram      {GPU_VRAM_GB:.1f} GB")
print(f"compute   {GPU_CAPABILITY[0]}.{GPU_CAPABILITY[1]}")
print(f"bf16      {'yes' if SUPPORTS_BF16 else 'no, so this run trains in fp16'}")

import accelerate  # noqa: E402 - pip ran above
import bitsandbytes  # noqa: E402 - pip ran above
import datasets  # noqa: E402 - pip ran above
import peft  # noqa: E402 - pip ran above
import transformers  # noqa: E402 - pip ran above
import trl  # noqa: E402 - pip ran above

VERSIONS = {
    "torch": torch.__version__,
    "transformers": transformers.__version__,
    "peft": peft.__version__,
    "trl": trl.__version__,
    "bitsandbytes": bitsandbytes.__version__,
    "accelerate": accelerate.__version__,
    "datasets": datasets.__version__,
}
print()
for name, version in VERSIONS.items():
    print(f"{name:<16} {version}")

In [ ]:
# ---------------------------------------------------------------------------
# 2. Clone the repository and read the config.
# ---------------------------------------------------------------------------
# The corpus, the reference set and every knob live in the repository. It is
# public, so no token is needed to read it.

from pathlib import Path

REPO_URL = "https://github.com/miztiik/yen-idhazh.git"
REPO_REF = "main"  # a branch, a tag or a commit sha
REPO_DIR = Path("/content/yen-idhazh" if IN_COLAB else "./yen-idhazh")

if REPO_DIR.exists():
    print(f"{REPO_DIR} is already here, fetching instead of cloning")
    fetch = ["git", "-C", str(REPO_DIR), "fetch", "--depth", "1", "origin", REPO_REF]
    subprocess.run(fetch, check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "checkout", "--force", "FETCH_HEAD"], check=True)
else:
    subprocess.run(
        ["git", "clone", "--depth", "1", "--branch", REPO_REF, REPO_URL, str(REPO_DIR)],
        check=True,
    )

REPO_SHA = subprocess.run(
    ["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"],
    check=True,
    capture_output=True,
    text=True,
).stdout.strip()

CONFIG = json.loads((REPO_DIR / "config" / "idhazh.json").read_text(encoding="utf-8"))
FINETUNE = CONFIG["finetune"]
MODELS = CONFIG["models"]

CORPUS_DIR = REPO_DIR / "corpus"
REFERENCE_DIR = REPO_DIR / "tests" / "fixtures" / "reference"

print(f"repo      {REPO_URL}")
print(f"ref       {REPO_REF}")
print(f"commit    {REPO_SHA}")
print(f"config    version {CONFIG['version']}")
print()
print("finetune block:")
for key, value in sorted(FINETUNE.items()):
    print(f"  {key:<22} {value}")

In [ ]:
# ---------------------------------------------------------------------------
# 3. Every knob, in one cell.
# ---------------------------------------------------------------------------
# Anything you would reach for is here. Nothing below this cell hides a number
# you might want to change.
#
# The values that describe the JOB come from config, so this notebook cannot
# drift from what production runs. The values that describe the MACHINE are
# here, because config knows nothing about which GPU you were given.

# --- from config. Do not hardcode these; change config/idhazh.json instead. ---
TEACHER_KEY = FINETUNE["teacher"]  # a key in `models`, never a model name
TRAIN_ROWS = FINETUNE["train_rows"]  # what one session samples
MIN_ROWS = FINETUNE["min_rows"]  # below this, nothing trains
EPOCHS = FINETUNE["epochs"]
SEQUENCE_LENGTH = FINETUNE["sequence_length"]

# --- LoRA. Estimates until a run measures them (Rule #10). ---
LORA_RANK = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.05
# All linear layers, because the MLP blocks carry format and format is most of
# what is being taught here. No embedding or head training.
LORA_TARGETS = [
    "q_proj", "k_proj", "v_proj", "o_proj",
    "gate_proj", "up_proj", "down_proj",
]

# --- optimiser ---
LEARNING_RATE = 2e-4
LR_SCHEDULE = "cosine"
WARMUP_RATIO = 0.03
MAX_GRAD_NORM = 1.0  # fp16 needs this; a Turing card will otherwise find a NaN
WEIGHT_DECAY = 0.0

# --- what fits on the card ---
BATCH_SIZE = 1
GRAD_ACCUM = 16  # effective batch 16
GRADIENT_CHECKPOINTING = True
# Turing has no bf16. Anything newer may use it.
USE_BF16 = SUPPORTS_BF16
USE_FP16 = not SUPPORTS_BF16

# Only touch this if the card runs out of memory. Rows longer than the value in
# force are DROPPED and counted, never truncated: a truncated target teaches the
# model to stop mid-summary, which is the one failure the reference set exists
# to prevent. `None` means use `finetune.sequence_length`.
SEQUENCE_LENGTH_OVERRIDE = None

# --- session hygiene ---
SEED = 0
CHECKPOINT_TO_DRIVE = IN_COLAB  # a dropped free session then costs minutes
DRIVE_MOUNT = Path("/content/drive")
CHECKPOINT_EVERY_MINUTES = 20
OUTPUT_DIR = Path("/content/adapter" if IN_COLAB else "./adapter")

# --- what to do with the result ---
DOWNLOAD_ADAPTER = True  # save it to your machine at the end
HF_UPLOAD_REPO = ""  # e.g. "your-name/yen-idhazh-summarizer-lora". Empty = skip.
HF_UPLOAD_PRIVATE = True

EFFECTIVE_SEQUENCE_LENGTH = SEQUENCE_LENGTH_OVERRIDE or SEQUENCE_LENGTH

print(f"teacher key            {TEACHER_KEY}")
print(f"train rows             {TRAIN_ROWS}")
print(f"epochs                 {EPOCHS}")
print(f"sequence length        {EFFECTIVE_SEQUENCE_LENGTH}", end="")
if SEQUENCE_LENGTH_OVERRIDE is None:
    print(" (config)")
else:
    print(f" (override; config says {SEQUENCE_LENGTH})")
print(f"effective batch        {BATCH_SIZE * GRAD_ACCUM}")
print(f"precision              {'bf16' if USE_BF16 else 'fp16'}")
print(f"lora                   r={LORA_RANK} alpha={LORA_ALPHA} dropout={LORA_DROPOUT}")

In [ ]:
# ---------------------------------------------------------------------------
# 4. Resolve the teacher, and stop if it is not the model production serves.
# ---------------------------------------------------------------------------
# No model is named in this notebook. `finetune.teacher` names a key in
# `models`; that entry carries both the GGUF the pipeline runs and the
# safetensors repo to train against. Holding them on one entry is deliberate:
# split apart, someone swaps the served model, forgets the other string, and
# this notebook trains an adapter against a different base. LoRA weights load
# onto a mismatched base without raising, so nothing would catch it.

from huggingface_hub import HfApi, hf_hub_download
from huggingface_hub.utils import (
    EntryNotFoundError,
    GatedRepoError,
    RepositoryNotFoundError,
)

if TEACHER_KEY not in MODELS:
    raise SystemExit(
        f"finetune.teacher is {TEACHER_KEY!r}, which is not a key in `models`. "
        f"Keys present: {sorted(k for k in MODELS if k != 'inference')}"
    )

TEACHER = MODELS[TEACHER_KEY]
BASE_REPO = TEACHER.get("hf_base_repo")
GGUF_REPO = TEACHER.get("repo")
GGUF_FILE = TEACHER.get("file")
GGUF_REVISION = TEACHER.get("revision")

if not BASE_REPO:
    raise SystemExit(
        f"models.{TEACHER_KEY}.hf_base_repo is not set, so there is nothing to train."
    )

api = HfApi()

# Gate one: the base repo has to exist.
try:
    base_info = api.model_info(BASE_REPO)
except RepositoryNotFoundError:
    raise SystemExit(
        f"models.{TEACHER_KEY}.hf_base_repo is {BASE_REPO!r} and Hugging Face has no "
        f"such repository. Fix the value in config/idhazh.json. Do not guess a "
        f"replacement here - the served GGUF and the trained base must be the same weights."
    ) from None
except GatedRepoError:
    raise SystemExit(
        f"{BASE_REPO} is gated. Accept its licence on huggingface.co, then set an "
        f"HF token in this runtime and re-run."
    ) from None

# Gate two: the GGUF the pipeline actually serves has to exist at the pinned revision.
try:
    api.model_info(GGUF_REPO, revision=GGUF_REVISION, files_metadata=False)
    gguf_files = api.list_repo_files(GGUF_REPO, revision=GGUF_REVISION)
except RepositoryNotFoundError:
    raise SystemExit(f"models.{TEACHER_KEY}.repo is {GGUF_REPO!r} and it does not exist.") from None

if GGUF_FILE not in gguf_files:
    raise SystemExit(
        f"{GGUF_FILE} is not in {GGUF_REPO} at revision {GGUF_REVISION[:12]}. "
        f"The config points at a file that is not there."
    )

# Gate three, and the one that actually catches a mismatched base: the base
# repo's own config, printed so a person can see the family and the size.
try:
    base_config = json.loads(
        Path(hf_hub_download(BASE_REPO, "config.json")).read_text(encoding="utf-8")
    )
except EntryNotFoundError:
    raise SystemExit(f"{BASE_REPO} has no config.json, so it is not a trainable base.") from None

print(f"teacher key       {TEACHER_KEY}")
print(f"base (train on)   {BASE_REPO}")
print(f"gguf (served)     {GGUF_REPO}/{GGUF_FILE} @ {GGUF_REVISION[:12]}")
print()
print(f"model_type        {base_config.get('model_type')}")
print(f"architectures     {base_config.get('architectures')}")
print(f"hidden_size       {base_config.get('hidden_size')}")
print(f"layers            {base_config.get('num_hidden_layers')}")
print(f"vocab_size        {base_config.get('vocab_size')}")
print(f"max_position      {base_config.get('max_position_embeddings')}")
print(f"downloads         {base_info.downloads:,}")
print(f"last modified     {base_info.lastModified}")

if base_config.get("max_position_embeddings", 0) < EFFECTIVE_SEQUENCE_LENGTH:
    raise SystemExit(
        f"The base supports {base_config.get('max_position_embeddings')} positions and "
        f"this run asks for {EFFECTIVE_SEQUENCE_LENGTH}. Lower "
        f"SEQUENCE_LENGTH_OVERRIDE or raise finetune.sequence_length's justification."
    )

In [ ]:
# ---------------------------------------------------------------------------
# 5. Load the rows, and refuse every one we are not allowed to train on.
# ---------------------------------------------------------------------------
# Four separate refusals, and each one is a real hazard rather than hygiene:
#
#   1. The corpus holdout. Held out by DATE, so the model trains on the past and
#      is measured on the future - which is the only split that resembles
#      production, where every article is one nobody has seen.
#   2. The reference set's test slice. Training on it would make the one honest
#      measurement of this work meaningless.
#   3. Any row whose assistant turn carries an injection marker. That turn is our
#      own model's output on a stranger's web page. If an attack ever landed in
#      production, the row would teach the tuned model the injected behaviour.
#   4. Any row longer than the sequence length. Dropped and counted, never cut.
#
# Every one raises. None warns.

import hashlib
import random
from collections import Counter, defaultdict


def read_jsonl(path):
    rows = []
    with path.open(encoding="utf-8") as handle:
        for line in handle:
            if line.strip():
                rows.append(json.loads(line))
    return rows


corpus_rows = read_jsonl(CORPUS_DIR / "corpus.jsonl")
reference_rows = read_jsonl(REFERENCE_DIR / "reference.jsonl")

holdout_path = CORPUS_DIR / "holdout.txt"
holdout = frozenset(
    line.strip() for line in holdout_path.read_text(encoding="utf-8").splitlines() if line.strip()
) if holdout_path.exists() else frozenset()

if len(corpus_rows) < MIN_ROWS:
    raise SystemExit(
        f"The corpus holds {len(corpus_rows)} rows and finetune.min_rows is {MIN_ROWS}. "
        f"Run the harvest, or `backend/utilities/data_wrangler.py refill`, before training."
    )

if not holdout:
    raise SystemExit(
        "corpus/holdout.txt is missing or empty. Run "
        "`python backend/utilities/data_wrangler.py split` and commit it. Training "
        "without a holdout produces a model nothing can measure."
    )

# The reference set's emitted file carries no slice, so the slice is recovered
# from the queue that produced it. A reference row whose side of the line cannot
# be established is refused rather than assumed into training.
queue_rows = read_jsonl(REFERENCE_DIR / "queue.jsonl")
slice_of = {row["url_key"]: row["slice"] for row in queue_rows}

unknown_slice = [row["url_key"] for row in reference_rows if row["url_key"] not in slice_of]
if unknown_slice:
    raise SystemExit(
        f"{len(unknown_slice)} reference rows are not in queue.jsonl, so nothing says "
        f"whether they are training targets or test references. First: "
        f"{unknown_slice[0][:12]}. Re-run `reference_set.py check --write`."
    )

reference_test = {row["url_key"] for row in reference_rows if slice_of[row["url_key"]] == "test"}
print(f"corpus rows            {len(corpus_rows)}")
print(f"reference rows         {len(reference_rows)}")
print(f"  of them the test slice {len(reference_test)}")
print(f"corpus holdout         {len(holdout)}")

# --- refusal 3: the injection markers, read from the canary suite itself ------
canary_markers = set()
for path in sorted((REPO_DIR / "tests" / "fixtures" / "canaries").rglob("*.json")):
    canary_markers.update(json.loads(path.read_text(encoding="utf-8")).get("must_not_survive", []))
print(f"injection markers      {len(canary_markers)} strings, from the canary suite")


def assistant_turn(row):
    for turn in row["messages"]:
        if turn["role"] == "assistant":
            return turn["content"]
    raise ValueError(f"{row['url_key'][:12]} has no assistant turn")


def carries_an_injection(row):
    text = assistant_turn(row)
    return any(marker in text for marker in canary_markers)


# --- assemble the pool -------------------------------------------------------
pool = []
dropped = Counter()

for row in corpus_rows:
    if row["url_key"] in holdout:
        dropped["in the corpus holdout"] += 1
    elif carries_an_injection(row):
        dropped["carries an injection marker"] += 1
    else:
        pool.append(dict(row, source="corpus"))

for row in reference_rows:
    if slice_of[row["url_key"]] == "test":
        dropped["the reference test slice"] += 1
    elif row["url_key"] in holdout:
        dropped["in the corpus holdout"] += 1
    elif carries_an_injection(row):
        dropped["carries an injection marker"] += 1
    else:
        pool.append(dict(row, source="reference"))

seen = set()
deduped = []
for row in pool:
    if row["url_key"] in seen:
        dropped["a duplicate url_key"] += 1
        continue
    seen.add(row["url_key"])
    deduped.append(row)
pool = deduped

print("\nrefused before sampling:")
for reason, count in sorted(dropped.items(), key=lambda item: -item[1]):
    print(f"  {count:>6}  {reason}")
print(f"\ntrainable pool         {len(pool)}")

if len(pool) < MIN_ROWS:
    raise SystemExit(
        f"Only {len(pool)} rows survive the refusals, against a floor of {MIN_ROWS}."
    )

In [ ]:
# ---------------------------------------------------------------------------
# 6. Sample, with a quota so one vertical cannot dominate.
# ---------------------------------------------------------------------------
# The window is deliberately twice what a session draws, and this is what that
# buys: a sample that is not just the loudest vertical. Every row a hand wrote
# is taken first - they are the ones the whole reference set was built to add.

random.seed(SEED)

by_vertical = defaultdict(list)
for row in pool:
    by_vertical[row["vertical"]].append(row)
for rows in by_vertical.values():
    rows.sort(key=lambda row: row["url_key"])  # deterministic, no clock, no shuffle order
    rows.sort(key=lambda row: 0 if row["source"] == "reference" else 1)

quota = -(-TRAIN_ROWS // max(1, len(by_vertical)))  # ceiling division
sampled, leftovers = [], []
for _, rows in sorted(by_vertical.items()):
    sampled.extend(rows[:quota])
    leftovers.extend(rows[quota:])

# Top up to TRAIN_ROWS from whatever the quota left behind, so a thin vertical
# costs the sample size rather than silently shrinking it.
leftovers.sort(key=lambda row: (0 if row["source"] == "reference" else 1, row["url_key"]))
sampled.extend(leftovers[: max(0, TRAIN_ROWS - len(sampled))])
sampled = sampled[:TRAIN_ROWS]
random.shuffle(sampled)

# The gate, restated as an assertion because a warning here is worthless.
leaked = {row["url_key"] for row in sampled} & holdout
if leaked:
    raise SystemExit(f"{len(leaked)} sampled rows are in the holdout: {sorted(leaked)[:3]}")
leaked_reference = {row["url_key"] for row in sampled} & reference_test
if leaked_reference:
    raise SystemExit(
        f"{len(leaked_reference)} sampled rows are reference TEST rows: "
        f"{sorted(leaked_reference)[:3]}"
    )

print(f"sampled                {len(sampled)} of a {TRAIN_ROWS}-row target")
print(f"quota per vertical     {quota}")
print()
for vertical, count in sorted(Counter(row["vertical"] for row in sampled).items()):
    print(f"  {vertical:<20} {count}")
print()
for source, count in sorted(Counter(row["source"] for row in sampled).items()):
    print(f"  from {source:<15} {count}")
print()
print("disjoint from the corpus holdout: yes")
print("disjoint from the reference test slice: yes")

In [ ]:
# ---------------------------------------------------------------------------
# 7. Tokenize, and assert the loss mask before anything trains.
# ---------------------------------------------------------------------------
# THIS IS THE CELL THAT MATTERS MOST.
#
# The median article here is about six times the length of its summary. Train on
# the whole sequence and roughly six sevenths of the gradient goes into learning
# to write other people's news articles, which is not the job.
#
# The usual fix passes the assistant header to a collator as a STRING. The
# tokenizer can split that string differently mid-sequence, the collator then
# matches nothing, masks nothing, and raises nothing. So the mask is built here
# by length - tokenize the prompt, tokenize the whole thing, mask the difference -
# and then asserted by decoding it back. If the unmasked span is not the JSON the
# model is supposed to write, this cell stops.

from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(BASE_REPO, trust_remote_code=False)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

IGNORE = -100


def encode(row):
    messages = row["messages"]
    prompt_messages = [turn for turn in messages if turn["role"] != "assistant"]

    prompt_ids = tokenizer.apply_chat_template(
        prompt_messages, tokenize=True, add_generation_prompt=True
    )
    full_ids = tokenizer.apply_chat_template(messages, tokenize=True, add_generation_prompt=False)

    # apply_chat_template must be a pure extension here, or the mask is a lie.
    if full_ids[: len(prompt_ids)] != prompt_ids:
        raise SystemExit(
            "The chat template does not extend the prompt when the assistant turn is "
            "added, so masking by length would mask the wrong tokens. This base's "
            "template is not the one the corpus was built with."
        )

    labels = [IGNORE] * len(prompt_ids) + full_ids[len(prompt_ids) :]
    return {"input_ids": full_ids, "labels": labels, "prompt_len": len(prompt_ids)}


encoded, too_long = [], []
for row in sampled:
    item = encode(row)
    if len(item["input_ids"]) > EFFECTIVE_SEQUENCE_LENGTH:
        too_long.append((len(item["input_ids"]), row["url_key"]))
        continue
    encoded.append(item)

if too_long:
    too_long.sort(reverse=True)
    print(
        f"dropped {len(too_long)} rows longer than {EFFECTIVE_SEQUENCE_LENGTH} tokens "
        f"(longest {too_long[0][0]}). Dropped, not truncated."
    )

lengths = sorted(len(item["input_ids"]) for item in encoded)
print(f"training rows          {len(encoded)}")
print(
    f"tokens per row         min {lengths[0]}, median {lengths[len(lengths) // 2]}, "
    f"max {lengths[-1]}"
)

# --- the oracle --------------------------------------------------------------
sample = encoded[0]
unmasked = [token for token in sample["labels"] if token != IGNORE]
learned_text = tokenizer.decode(unmasked, skip_special_tokens=True).strip()
prompt_ids = sample["input_ids"][: sample["prompt_len"]]
masked_text = tokenizer.decode(prompt_ids, skip_special_tokens=True)

print("\n--- what this run will LEARN TO WRITE (labels != -100) ---")
print(learned_text[:600])
print("\n--- what it will only READ (masked) ---")
print(masked_text[:300].replace("\n", " ") + " ...")

share = len(unmasked) / len(sample["input_ids"])
print(f"\nunmasked share of the sequence: {share:.1%}")

parsed = json.loads(learned_text)
missing = {"title", "summary", "key_points"} - set(parsed)
if missing:
    raise SystemExit(f"The unmasked span parses but is missing {sorted(missing)}.")
if share > 0.5:
    raise SystemExit(
        f"{share:.0%} of the sequence is unmasked. That is the article, not the summary."
    )
print("loss mask asserted: the unmasked span is the assistant turn's JSON, and only that.")

In [ ]:
# ---------------------------------------------------------------------------
# 8. Train.
# ---------------------------------------------------------------------------
# QLoRA, not a full fine-tune: the base already summarizes, and what it needs is
# our house rules - a small change to a large model.
#
# No example packing, and sdpa rather than FlashAttention-2. Packing crosses
# article boundaries, and FA2 needs Ampere or newer, so a T4 cannot stop packed
# examples from attending to each other.

from datasets import Dataset
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from transformers import (
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    Trainer,
    TrainingArguments,
    set_seed,
)

set_seed(SEED)

if CHECKPOINT_TO_DRIVE:
    from google.colab import drive

    drive.mount(str(DRIVE_MOUNT), force_remount=False)
    checkpoint_dir = DRIVE_MOUNT / "MyDrive" / "yen-idhazh" / "adapter"
    checkpoint_dir.mkdir(parents=True, exist_ok=True)
else:
    checkpoint_dir = OUTPUT_DIR
    checkpoint_dir.mkdir(parents=True, exist_ok=True)
print(f"checkpoints -> {checkpoint_dir}")


def collate(batch):
    width = max(len(item["input_ids"]) for item in batch)
    pad = tokenizer.pad_token_id
    return {
        "input_ids": torch.tensor(
            [item["input_ids"] + [pad] * (width - len(item["input_ids"])) for item in batch]
        ),
        "labels": torch.tensor(
            [item["labels"] + [IGNORE] * (width - len(item["labels"])) for item in batch]
        ),
        "attention_mask": torch.tensor(
            [
                [1] * len(item["input_ids"]) + [0] * (width - len(item["input_ids"]))
                for item in batch
            ]
        ),
    }


dataset = Dataset.from_list([{k: item[k] for k in ("input_ids", "labels")} for item in encoded])

model = AutoModelForCausalLM.from_pretrained(
    BASE_REPO,
    quantization_config=BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=torch.bfloat16 if USE_BF16 else torch.float16,
    ),
    attn_implementation="sdpa",
    device_map={"": 0},
    torch_dtype=torch.bfloat16 if USE_BF16 else torch.float16,
)
model.config.use_cache = False
model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=GRADIENT_CHECKPOINTING)
model = get_peft_model(
    model,
    LoraConfig(
        r=LORA_RANK,
        lora_alpha=LORA_ALPHA,
        lora_dropout=LORA_DROPOUT,
        target_modules=LORA_TARGETS,
        bias="none",
        task_type="CAUSAL_LM",
    ),
)
model.print_trainable_parameters()

steps_per_epoch = max(1, len(dataset) // (BATCH_SIZE * GRAD_ACCUM))
save_steps = max(10, steps_per_epoch // 4)

trainer = Trainer(
    model=model,
    train_dataset=dataset,
    data_collator=collate,
    args=TrainingArguments(
        output_dir=str(checkpoint_dir),
        num_train_epochs=EPOCHS,
        per_device_train_batch_size=BATCH_SIZE,
        gradient_accumulation_steps=GRAD_ACCUM,
        gradient_checkpointing=GRADIENT_CHECKPOINTING,
        learning_rate=LEARNING_RATE,
        lr_scheduler_type=LR_SCHEDULE,
        warmup_ratio=WARMUP_RATIO,
        max_grad_norm=MAX_GRAD_NORM,
        weight_decay=WEIGHT_DECAY,
        bf16=USE_BF16,
        fp16=USE_FP16,
        optim="paged_adamw_8bit",
        logging_steps=10,
        save_steps=save_steps,
        save_total_limit=2,
        report_to=[],
        seed=SEED,
    ),
)

print(f"\nsteps per epoch        {steps_per_epoch}")
print(f"checkpoint every       {save_steps} steps")
print(f"precision              {'bf16' if USE_BF16 else 'fp16'}\n")

TRAIN_STARTED = time.time()
result = trainer.train()
TRAIN_SECONDS = time.time() - TRAIN_STARTED

final_loss = result.training_loss
print(f"\nwall clock             {TRAIN_SECONDS / 60:.1f} min")
print(f"final training loss    {final_loss:.4f}")
print(f"peak vram              {torch.cuda.max_memory_allocated() / 1024**3:.2f} GB")

if final_loss != final_loss:  # NaN, which fp16 on a Turing card can produce
    raise SystemExit(
        "The training loss is NaN. fp16 on a Turing card overflowed. Lower "
        "LEARNING_RATE, confirm MAX_GRAD_NORM is 1.0, and run again."
    )

In [ ]:
# ---------------------------------------------------------------------------
# 9. Showcase: what does it write now, against what the base wrote?
# ---------------------------------------------------------------------------
# On held-out rows only - articles neither the base nor this adapter was trained
# on. This is a look, not a measurement. The measurement is two independent
# scorers and a blind read, and it happens back in the repository.

from transformers import GenerationConfig

held_out_rows = [row for row in corpus_rows if row["url_key"] in holdout][:3]
model.eval()
model.config.use_cache = True

generation = GenerationConfig(
    max_new_tokens=CONFIG["models"]["inference"]["max_output_tokens"],
    do_sample=False,
    temperature=None,
    top_p=None,
    pad_token_id=tokenizer.pad_token_id,
)


def write_summary(row, adapter_on):
    prompt = [turn for turn in row["messages"] if turn["role"] != "assistant"]
    ids = tokenizer.apply_chat_template(
        prompt, tokenize=True, add_generation_prompt=True, return_tensors="pt"
    ).to(model.device)
    context = model.disable_adapter() if not adapter_on else torch.no_grad()
    with torch.no_grad(), context:
        out = model.generate(input_ids=ids, generation_config=generation)
    return tokenizer.decode(out[0][ids.shape[-1] :], skip_special_tokens=True).strip()


def show(label, text):
    print(f"  {label}")
    try:
        parsed = json.loads(text)
        print(f"    title   {parsed.get('title')}")
        summary = parsed.get("summary", "")
        print(f"    words   {len(summary.split())}")
        print(f"    points  {len(parsed.get('key_points', []))}")
        print(f"    summary {summary[:220]}...")
    except json.JSONDecodeError:
        print(f"    DID NOT PARSE AS JSON: {text[:220]}")


for row in held_out_rows:
    article = next(turn["content"] for turn in row["messages"] if turn["role"] == "user")
    print("=" * 78)
    print(f"{row['date']}  {row['vertical']}  {row['url_key'][:12]}")
    print(f"article starts: {article[:150].replace(chr(10), ' ')} ...")
    print()
    show("BASE (adapter off)", write_summary(row, adapter_on=False))
    print()
    show("TUNED (adapter on)", write_summary(row, adapter_on=True))
    print()
    show("WHAT THE PIPELINE PUBLISHED", assistant_turn(row))
    print()

model.config.use_cache = False

In [ ]:
# ---------------------------------------------------------------------------
# 10. Save the adapter, and record what this run actually cost.
# ---------------------------------------------------------------------------
# Rule #10: a number without its hardware and its date is not a measurement.
# Paste the block this prints into docs/reference/measurements.md.

import datetime
import shutil

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
model.save_pretrained(str(OUTPUT_DIR))
tokenizer.save_pretrained(str(OUTPUT_DIR))

adapter_file = OUTPUT_DIR / "adapter_model.safetensors"
if not adapter_file.exists():
    adapter_file = OUTPUT_DIR / "adapter_model.bin"

ADAPTER_BYTES = adapter_file.stat().st_size
ADAPTER_SHA256 = hashlib.sha256(adapter_file.read_bytes()).hexdigest()
TOTAL_SECONDS = time.time() - RUN_STARTED

RUN_RECORD = {
    "date": datetime.date.today().isoformat(),
    "repo_commit": REPO_SHA,
    "config_version": CONFIG["version"],
    "teacher_key": TEACHER_KEY,
    "base_repo": BASE_REPO,
    "gpu": GPU_NAME,
    "vram_gb": round(GPU_VRAM_GB, 1),
    "precision": "bf16" if USE_BF16 else "fp16",
    "rows_trained": len(encoded),
    "rows_dropped_too_long": len(too_long),
    "epochs": EPOCHS,
    "sequence_length": EFFECTIVE_SEQUENCE_LENGTH,
    "effective_batch": BATCH_SIZE * GRAD_ACCUM,
    "lora": {"r": LORA_RANK, "alpha": LORA_ALPHA, "dropout": LORA_DROPOUT},
    "learning_rate": LEARNING_RATE,
    "train_minutes": round(TRAIN_SECONDS / 60, 1),
    "session_minutes": round(TOTAL_SECONDS / 60, 1),
    "peak_vram_gb": round(torch.cuda.max_memory_allocated() / 1024**3, 2),
    "final_training_loss": round(float(final_loss), 4),
    "adapter_sha256": ADAPTER_SHA256,
    "adapter_bytes": ADAPTER_BYTES,
    "versions": VERSIONS,
}
record_json = json.dumps(RUN_RECORD, indent=2, sort_keys=True)
(OUTPUT_DIR / "run.json").write_text(record_json, encoding="utf-8")

print(f"adapter        {adapter_file.name}")
print(f"bytes          {ADAPTER_BYTES:,}")
print(f"sha256         {ADAPTER_SHA256}")
print(f"train minutes  {TRAIN_SECONDS / 60:.1f}")
print()
print("--- paste into docs/reference/measurements.md ---")
print(json.dumps(RUN_RECORD, indent=2, sort_keys=True))

ARCHIVE = shutil.make_archive(str(OUTPUT_DIR), "zip", str(OUTPUT_DIR))
print(f"\nzipped to {ARCHIVE} ({Path(ARCHIVE).stat().st_size:,} bytes)")

In [ ]:
# ---------------------------------------------------------------------------
# 11. Take it with you: download, or push to Hugging Face.
# ---------------------------------------------------------------------------
# The adapter is a few tens of megabytes. The merged model is not, which is why
# the merge happens on your machine in the next cell's instructions.
#
# To upload instead of download, set HF_UPLOAD_REPO in cell 3 and put a WRITE
# token in this runtime. In Colab: the key icon in the left sidebar, name it
# HF_TOKEN, and switch on "Notebook access". Never paste a token into a cell -
# this notebook is committed to a public repository.

token = None
if IN_COLAB:
    try:
        from google.colab import userdata

        token = userdata.get("HF_TOKEN")
    except Exception:
        token = None
token = token or os.environ.get("HF_TOKEN")

if HF_UPLOAD_REPO:
    if not token:
        raise SystemExit(
            "HF_UPLOAD_REPO is set but no token was found. Add HF_TOKEN as a Colab "
            "secret with notebook access, or export HF_TOKEN, then run this cell again."
        )
    api = HfApi(token=token)
    api.create_repo(HF_UPLOAD_REPO, private=HF_UPLOAD_PRIVATE, exist_ok=True)
    api.upload_folder(
        folder_path=str(OUTPUT_DIR),
        repo_id=HF_UPLOAD_REPO,
        commit_message=(
            f"LoRA adapter for {TEACHER_KEY}, {len(encoded)} rows, "
            f"{EPOCHS} epochs, sha256 {ADAPTER_SHA256[:12]}"
        ),
    )
    print(f"pushed to https://huggingface.co/{HF_UPLOAD_REPO}")
    print(f"private: {HF_UPLOAD_PRIVATE}")
else:
    print("HF_UPLOAD_REPO is empty, so nothing was uploaded.")

if DOWNLOAD_ADAPTER and IN_COLAB:
    from google.colab import files

    print("\nyour browser will now download the adapter zip ...")
    files.download(ARCHIVE)
elif DOWNLOAD_ADAPTER:
    print(f"\nnot in Colab. The adapter is at {ARCHIVE}")

## What to do next, on your machine

The merge and the quantise do not happen here. Loading the whole model at
16-bit to merge wants about 16 GB of ordinary RAM against the free tier's 12,
so training would succeed and the save would die. llama.cpp streams from disk
instead, and it is already in `backend/bin/`.

**1. Unzip the adapter** next to the repository.

**2. Convert the adapter to GGUF.** From a clone of `ggml-org/llama.cpp`:

```bash
python convert_lora_to_gguf.py /path/to/adapter --outfile adapter-f16.gguf --outtype f16
```

**3. Merge it into the base GGUF the pipeline already serves.** The base is the
file `models.<teacher>.file` names, which is already in `backend/models/`:

```bash
./llama-export-lora \
  -m backend/models/<the file config names> \
  -o backend/models/<a new name>-merged.gguf \
  --lora adapter-f16.gguf
```

**4. Quantise to the quantisation config asks for:**

```bash
./llama-quantize backend/models/<a new name>-merged.gguf \
                 backend/models/<a new name>-Q4_K_M.gguf Q4_K_M
```

**5. Take its SHA-256 and its byte count.** Read the hash from the git-LFS
pointer at the commit, never from the resolve URL's `ETag` - that is a Xet
content hash. It is 64 hex characters, it looks exactly like a SHA-256, and it
is not one. It bites hardest on your own upload, because your own upload feels
trustworthy.

```bash
sha256sum backend/models/<a new name>-Q4_K_M.gguf
stat -c %s backend/models/<a new name>-Q4_K_M.gguf
```

**6. Qualify it like any other candidate.** A fine-tuned model is not special:
one entry in `config/idhazh.json`, the same qualification, the same SHA-256.

**Expect `injection_canaries` to be the gate that fails.** The safety clauses
live in the masked system prompt and are never trained on, so a model tuned to
emit a shape can stop attending to them. The canaries run live, which is
exactly why that is caught rather than shipped.

**Adopting the result is a separate decision and needs explicit approval.**
Nothing here changes what a reader receives.

## See also

- `docs/how-to/fine-tune-a-model.md` - the corpus, the three verbs, the knobs.
- `TODO/20260827-summarizer-fine-tuning-plan.md` - rows 7 to 10, and why each
  decision here is the one it is.